
<div  style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Associate/main/Includes/images/bookstore_schema.png" alt="Databricks Learning" style="width: 600">
</div>

##Querying JSON

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
%python
files = dbutils.fs.ls(f"{dataset_bookstore}/customers-json")
display(files)

In [0]:
select * from json.`${dataset.bookstore}/customers-json/export_001.json`

In [0]:
select * from json.`${dataset.bookstore}/customers-json/export_*.json`

In [0]:
select * from json.`${dataset.bookstore}/customers-json`

In [0]:
select count(*) from json.`${dataset.bookstore}/customers-json`

In [0]:
select *,input_file_name() source_file
from json.`${dataset.bookstore}/customers-json` ---Input file name is a special Spark SQL command that records the source file for each record

## Querying text format

In [0]:
select * from text.`${dataset.bookstore}/customers-json`

##Querying Binary Format

In [0]:
select * from binaryFile.`${dataset.bookstore}/customers-json`

## Querying CSV

In [0]:
select * from csv.`${dataset.bookstore}/books-csv`

In [0]:
show external locations

In [0]:
%python
dbutils.widgets.text("external_location", 'abfss://unity-catalog-storage@dbstorage5xhceo3gfqey2.dfs.core.windows.net/7405612281606284/external_storage')

external_location = dbutils.widgets.get("external_location")
dbutils.fs.cp(f"{dataset_bookstore}/books-csv", f"{external_location}/books-csv", recurse=True)

In [0]:
create table books_csv
(book_id string,title string,author string,catagory string,price double)
using csv
options(header= "true",
        delimiter =";")
location "${external_location}/books-csv"

In [0]:
select * from books_csv

In [0]:
describe extended books_csv

In [0]:
%python
files = dbutils.fs.ls(f"{external_location}/books-csv")
display(files)

In [0]:
%python
(spark.read.table("books_csv")
 .write
 .mode("append")
 .format("csv")
 .option("header", "true")
 .option("delimiter",";")
 .save(f"{external_location}/books-csv"))

In [0]:
%python
files = dbutils.fs.ls(f"{external_location}/books-csv")
display(files)

In [0]:
SELECT COUNT(*) FROM books_csv

In [0]:
REFRESH TABLE books_csv

In [0]:
SELECT COUNT(*) FROM books_csv

##CTAS Statement

In [0]:
create table customers as
select * from json.`${dataset.bookstore}/customers-json`;

describe extended customers

In [0]:
create table books_unparsed as
select * from csv.`${dataset.bookstore}/books-csv`;

select * from books_unparsed

In [0]:
create temporary view books_tmp_vw
    (book_id string, title string, author string, category string, price double)
using csv
options(
    path = "${dataset.bookstore}/books-csv/export_*.csv",
    header = "true",
    sep = ";"
);

create table books as
select * from books_tmp_vw;

select * from books

In [0]:
describe extended books

In [0]:
drop table books_read_files

In [0]:
CREATE TABLE books_read_files
as SELECT * FROM read_files(
  "/Volumes/demoworkspace/default/bookstore_dataset/books-csv/export_*.csv",
  format => 'csv',
  header => 'true',
  delimiter => ';')

In [0]:
select * from books_read_files

##_metadata Column

In [0]:
select *,_metadata.file_path,_metadata.file_modification_time,_metadata.file_size,_metadata.file_name
from json.`/Volumes/demoworkspace/default/bookstore_dataset/customers-json` ---_metadata function is a special Spark SQL command that records the source file for each record